# LangChain & LangGraph Practical Assignment

**Program:** LangChain & LangGraph

This notebook covers the 20 practical questions given in the assignment. The code is kept simple and each question is separated so it is easy to run and explain.

## 0. Setup

Run this cell first. It installs the packages used in the notebook.

In [ ]:
!pip -q install -U langchain langchain-openai langchain-community langgraph faiss-cpu pypdf sentence-transformers wikipedia requests langchain-huggingface

## API Key

Enter the OpenAI API key before running the LLM based examples.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

print("Setup completed")

# 1. Prompt Template Chatbot

Create a chatbot using `PromptTemplate` that answers questions politely and professionally.

In [ ]:
prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are a polite and professional chatbot.
Answer the user's question in a simple and helpful way.

Question: {question}
Answer:"""
)

chatbot = prompt | llm

question = input("Ask a question: ")
answer = chatbot.invoke({"question": question})

print(answer.content)

# 2. Personalized Roadmap Generator

Take the user's name, profession and learning goal and create a simple learning roadmap.

In [ ]:
name = input("Name: ")
profession = input("Profession: ")
goal = input("Learning goal: ")

roadmap_prompt = PromptTemplate(
    input_variables=["name", "profession", "goal"],
    template="""Create a practical learning roadmap for {name}.
Current profession: {profession}
Learning goal: {goal}

Give:
1. Topics to learn
2. Practice activities
3. A small project
4. A suggested order

Keep it simple and realistic."""
)

roadmap_chain = roadmap_prompt | llm
result = roadmap_chain.invoke({
    "name": name,
    "profession": profession,
    "goal": goal
})

print(result.content)

# 3. Sequential Content Processing Chain

The article is first summarized, then key points are extracted, and finally a short conclusion is generated.

In [ ]:
article = """
Artificial intelligence is being used in many industries such as healthcare,
banking and education. It can automate repetitive tasks and help people make
better decisions. However, organizations also need to consider data privacy,
security and responsible use while adopting AI.
"""

summary_prompt = PromptTemplate(
    input_variables=["article"],
    template="Summarize this article in 3-4 sentences:\n{article}"
)

key_prompt = PromptTemplate(
    input_variables=["summary"],
    template="Extract 4 important key points from this summary:\n{summary}"
)

conclusion_prompt = PromptTemplate(
    input_variables=["key_points"],
    template="Write a short conclusion based on these key points:\n{key_points}"
)

summary_chain = summary_prompt | llm
key_chain = key_prompt | llm
conclusion_chain = conclusion_prompt | llm

summary = summary_chain.invoke({"article": article}).content
key_points = key_chain.invoke({"summary": summary}).content
conclusion = conclusion_chain.invoke({"key_points": key_points}).content

print("SUMMARY:\n", summary)
print("\nKEY POINTS:\n", key_points)
print("\nCONCLUSION:\n", conclusion)

# 4. Translation and Sentiment Analysis

The input is translated to English and then classified as positive, negative or neutral.

In [ ]:
text = input("Enter text in any language: ")

translate_prompt = PromptTemplate(
    input_variables=["text"],
    template="Translate the following text into English. Return only the translation.\n{text}"
)

sentiment_prompt = PromptTemplate(
    input_variables=["text"],
    template="""Classify the sentiment as Positive, Negative, or Neutral.
Return the label and one short reason.

Text: {text}"""
)

english_text = (translate_prompt | llm).invoke({"text": text}).content
sentiment = (sentiment_prompt | llm).invoke({"text": english_text}).content

print("English:", english_text)
print("Sentiment:", sentiment)

# 5. PDF Question-Answering System

Upload a PDF and answer questions using only its contents.

In [ ]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

uploaded = files.upload()
pdf_path = next(iter(uploaded))

loader = PyPDFLoader(pdf_path)
documents = loader.load()

pdf_text = "\n".join(doc.page_content for doc in documents)

pdf_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""Answer the question using only the document text below.
If the answer is not present, say: "The answer is not available in the document."

Document:
{context}

Question:
{question}

Answer:"""
)

question = input("Ask a question about the PDF: ")
pdf_chain = pdf_prompt | llm

result = pdf_chain.invoke({
    "context": pdf_text[:30000],
    "question": question
})

print(result.content)

# 6. Resume Analysis Application

The program checks a resume for skills, missing skills, job roles and improvement suggestions.

In [ ]:
resume = input("Paste resume text here: ")

resume_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""Analyze the following resume.

Give these four sections:
1. Skills identified
2. Missing or useful skills
3. Suggested job roles
4. Resume improvement suggestions

Resume:
{resume}"""
)

result = (resume_prompt | llm).invoke({"resume": resume})
print(result.content)

# 7. Multi-document Chatbot

Upload multiple PDFs. The documents are stored separately and the model is given the most relevant document text.

In [ ]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

uploaded = files.upload()

all_docs = []

for file_name in uploaded:
    loader = PyPDFLoader(file_name)
    docs = loader.load()

    for doc in docs:
        doc.metadata["source_file"] = file_name
        all_docs.append(doc)

print("Documents loaded:", len(uploaded))

question = input("Ask a question: ")

# Simple keyword based selection keeps this example easy to understand.
question_words = set(question.lower().split())

scores = []
for doc in all_docs:
    words = set(doc.page_content.lower().split())
    score = len(question_words.intersection(words))
    scores.append((score, doc))

scores.sort(key=lambda x: x[0], reverse=True)

best_docs = [doc for score, doc in scores[:3]]

context = "\n\n".join(
    f"Source: {doc.metadata.get('source_file')}\n{doc.page_content}"
    for doc in best_docs
)

multi_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""Answer the question using the provided document content.
Mention the source file when possible.

Context:
{context}

Question:
{question}

Answer:"""
)

answer = (multi_prompt | llm).invoke({
    "context": context[:30000],
    "question": question
})

print(answer.content)

# 8. Tool-enabled AI Assistant

Three tools are created: calculator, date/time and Wikipedia search. The assistant decides which tool to use.

In [ ]:
from langchain.tools import tool
from datetime import datetime
import requests
import ast
import operator as op

@tool
def calculator(expression: str) -> str:
    """Calculate a basic arithmetic expression."""
    allowed = {
        ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
        ast.Div: op.truediv, ast.Pow: op.pow
    }

    def calc(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in allowed:
            return allowed[type(node.op)](calc(node.left), calc(node.right))
        raise ValueError("Only basic arithmetic is allowed")

    return str(calc(ast.parse(expression, mode="eval").body))

@tool
def date_time() -> str:
    """Return the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def wikipedia_search(query: str) -> str:
    """Search Wikipedia and return a short result."""
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + query.replace(" ", "_")
    response = requests.get(url, timeout=10)

    if response.status_code == 200:
        data = response.json()
        return data.get("extract", "No result found.")

    return "No Wikipedia result found."

tools = [calculator, date_time, wikipedia_search]
tool_map = {tool.name: tool for tool in tools}

assistant_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Decide which tool is needed.

Tools:
- calculator: for arithmetic
- date_time: for current date/time
- wikipedia_search: for general factual lookup

Return exactly:
TOOL: <tool name>
INPUT: <tool input>

Question: {question}"""
)

question = input("Ask the assistant: ")
decision = (assistant_prompt | llm).invoke({"question": question}).content

print("Decision:", decision)

tool_name = None
tool_input = ""

for line in decision.splitlines():
    if line.startswith("TOOL:"):
        tool_name = line.split(":", 1)[1].strip()
    elif line.startswith("INPUT:"):
        tool_input = line.split(":", 1)[1].strip()

if tool_name in tool_map:
    tool_result = tool_map[tool_name].invoke(tool_input)
    print("Tool result:", tool_result)
else:
    print("Tool was not selected correctly.")

# 9. Memory-based Chatbot

A small dictionary is used as memory for name, programming language and favorite topic.

In [ ]:
memory = {
    "name": "",
    "programming_language": "",
    "favorite_topic": ""
}

memory["name"] = input("What is your name? ")
memory["programming_language"] = input("Preferred programming language? ")
memory["favorite_topic"] = input("Favorite learning topic? ")

print("\nSaved information:")
print(memory)

memory_prompt = PromptTemplate(
    input_variables=["name", "language", "topic", "question"],
    template="""You are a friendly learning assistant.

User name: {name}
Preferred programming language: {language}
Favorite learning topic: {topic}

Answer this question while remembering the user's information:
{question}"""
)

question = input("\nAsk something: ")

result = (memory_prompt | llm).invoke({
    "name": memory["name"],
    "language": memory["programming_language"],
    "topic": memory["favorite_topic"],
    "question": question
})

print(result.content)

# 10. RAG Chatbot with FAISS

This follows the required flow: PDF loading → text splitting → embeddings → FAISS → retrieval → answer.

In [ ]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


uploaded = files.upload()
pdf_path = next(iter(uploaded))

loader = PyPDFLoader(pdf_path)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)
chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(chunks, embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

question = input("Ask a question: ")
retrieved_docs = retriever.invoke(question)

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""Answer using only the retrieved context.
If the answer is not available, say that it is not found in the document.

Context:
{context}

Question:
{question}

Answer:"""
)

answer = (rag_prompt | llm).invoke({
    "context": context,
    "question": question
})

print(answer.content)

# LangGraph Questions 11–20

The following sections use `StateGraph` with shared state. LangGraph workflows are built from nodes and edges and compiled before invocation.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("LangGraph imports completed")

# 11. Basic State Graph

Workflow: Input → Process → Output.

In [ ]:
class BasicState(TypedDict):
    message: str
    result: str

def input_node(state):
    return {"message": state["message"]}

def process_node(state):
    return {"result": state["message"].upper()}

def output_node(state):
    print("Output:", state["result"])
    return {}

builder = StateGraph(BasicState)
builder.add_node("input", input_node)
builder.add_node("process", process_node)
builder.add_node("output", output_node)

builder.add_edge(START, "input")
builder.add_edge("input", "process")
builder.add_edge("process", "output")
builder.add_edge("output", END)

graph = builder.compile()

graph.invoke({"message": "hello langgraph", "result": ""})

# 12. Multi-step Conversation Graph

Nodes: Greeting → Information Collection → Response → Goodbye. Conversation details are stored in state.

In [ ]:
class ConversationState(TypedDict):
    name: str
    topic: str
    response: str

def greeting(state):
    print("Hello! Welcome to the learning assistant.")
    return {}

def collect_info(state):
    return {
        "name": input("Your name: "),
        "topic": input("What do you want to learn? ")
    }

def response_node(state):
    text = f"Nice to meet you {state['name']}. We can work on {state['topic']}."
    print(text)
    return {"response": text}

def goodbye(state):
    print("Goodbye,", state["name"])

builder = StateGraph(ConversationState)
builder.add_node("greeting", greeting)
builder.add_node("collect", collect_info)
builder.add_node("response", response_node)
builder.add_node("goodbye", goodbye)

builder.add_edge(START, "greeting")
builder.add_edge("greeting", "collect")
builder.add_edge("collect", "response")
builder.add_edge("response", "goodbye")
builder.add_edge("goodbye", END)

conversation_graph = builder.compile()

conversation_graph.invoke({"name": "", "topic": "", "response": ""})

# 13. Conditional Routing Graph

The query is routed to Math, Coding or General based on simple keyword checks.

In [ ]:
class RouteState(TypedDict):
    query: str
    answer: str

def detect_type(state):
    query = state["query"].lower()

    if any(word in query for word in ["add", "multiply", "divide", "calculate", "+", "-", "*", "/"]):
        return "math"
    if any(word in query for word in ["python", "code", "program", "function"]):
        return "coding"
    return "general"

def math_node(state):
    return {"answer": "This query was routed to the Math node."}

def coding_node(state):
    return {"answer": "This query was routed to the Coding node."}

def general_node(state):
    return {"answer": "This query was routed to the General Questions node."}

builder = StateGraph(RouteState)
builder.add_node("math", math_node)
builder.add_node("coding", coding_node)
builder.add_node("general", general_node)

builder.add_conditional_edges(
    START,
    detect_type,
    {
        "math": "math",
        "coding": "coding",
        "general": "general"
    }
)

builder.add_edge("math", END)
builder.add_edge("coding", END)
builder.add_edge("general", END)

route_graph = builder.compile()

query = input("Enter a query: ")
result = route_graph.invoke({"query": query, "answer": ""})

print(result["answer"])

# 14. Intent-based Workflow

The first node detects intent, the second routes the request and the final node generates a response.

In [ ]:
class IntentState(TypedDict):
    query: str
    intent: str
    response: str

def detect_intent(state):
    q = state["query"].lower()

    if any(x in q for x in ["price", "cost", "payment"]):
        intent = "billing"
    elif any(x in q for x in ["error", "not working", "issue"]):
        intent = "technical"
    else:
        intent = "general"

    return {"intent": intent}

def route_request(state):
    return {}

def generate_response(state):
    prompt = PromptTemplate(
        input_variables=["intent", "query"],
        template="""Respond to this user request.
Intent: {intent}
Request: {query}
Give a short helpful response."""
    )

    answer = (prompt | llm).invoke({
        "intent": state["intent"],
        "query": state["query"]
    }).content

    return {"response": answer}

builder = StateGraph(IntentState)
builder.add_node("detect_intent", detect_intent)
builder.add_node("route", route_request)
builder.add_node("generate", generate_response)

builder.add_edge(START, "detect_intent")
builder.add_edge("detect_intent", "route")
builder.add_edge("route", "generate")
builder.add_edge("generate", END)

intent_graph = builder.compile()

query = input("Enter a request: ")
result = intent_graph.invoke({
    "query": query,
    "intent": "",
    "response": ""
})

print("Intent:", result["intent"])
print("Response:", result["response"])

# 15. Document Processing Workflow

Separate nodes are used for: Load PDF → Split text → Create embeddings → Retrieve → Generate answer.

In [ ]:
from typing import TypedDict, Any
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

class DocumentState(TypedDict):
    pdf_path: str
    question: str
    documents: Any
    chunks: Any
    vector_db: Any
    retrieved: Any
    answer: str

def load_pdf(state):
    docs = PyPDFLoader(state["pdf_path"]).load()
    return {"documents": docs}

def split_text(state):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )
    chunks = splitter.split_documents(state["documents"])
    return {"chunks": chunks}

def create_embeddings(state):
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    db = FAISS.from_documents(state["chunks"], embeddings)
    return {"vector_db": db}

def retrieve_content(state):
    docs = state["vector_db"].similarity_search(state["question"], k=3)
    return {"retrieved": docs}

def generate_answer(state):
    context = "\n\n".join(doc.page_content for doc in state["retrieved"])

    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template="""Answer only from the context.

Context:
{context}

Question:
{question}

Answer:"""
    )

    answer = (prompt | llm).invoke({
        "context": context,
        "question": state["question"]
    }).content

    return {"answer": answer}

builder = StateGraph(DocumentState)
builder.add_node("load_pdf", load_pdf)
builder.add_node("split_text", split_text)
builder.add_node("create_embeddings", create_embeddings)
builder.add_node("retrieve", retrieve_content)
builder.add_node("generate", generate_answer)

builder.add_edge(START, "load_pdf")
builder.add_edge("load_pdf", "split_text")
builder.add_edge("split_text", "create_embeddings")
builder.add_edge("create_embeddings", "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

document_graph = builder.compile()

uploaded = files.upload()
pdf_path = next(iter(uploaded))
question = input("Question about the PDF: ")

result = document_graph.invoke({
    "pdf_path": pdf_path,
    "question": question,
    "documents": [],
    "chunks": [],
    "vector_db": None,
    "retrieved": [],
    "answer": ""
})

print(result["answer"])

# 16. Error Handling Workflow

Empty or invalid input is sent to an error node. Valid input continues to the response node.

In [ ]:
class ErrorState(TypedDict):
    query: str
    message: str

def check_input(state):
    if not state["query"].strip():
        return "error"
    return "valid"

def error_node(state):
    return {"message": "Please enter a valid question."}

def valid_node(state):
    return {"message": "Valid input received: " + state["query"]}

builder = StateGraph(ErrorState)
builder.add_node("error", error_node)
builder.add_node("valid", valid_node)

builder.add_conditional_edges(
    START,
    check_input,
    {"error": "error", "valid": "valid"}
)

builder.add_edge("error", END)
builder.add_edge("valid", END)

error_graph = builder.compile()

query = input("Enter something: ")
result = error_graph.invoke({"query": query, "message": ""})

print(result["message"])

# 17. Approval Workflow Graph

Flow: Generate output → Review → Final output. A rejected result goes back for regeneration.

In [ ]:
class ApprovalState(TypedDict):
    topic: str
    draft: str
    approved: bool
    final: str

def generate_output(state):
    prompt = PromptTemplate(
        input_variables=["topic"],
        template="Write a short answer about: {topic}"
    )
    draft = (prompt | llm).invoke({"topic": state["topic"]}).content
    return {"draft": draft}

def review_output(state):
    print("\nGenerated output:\n", state["draft"])
    choice = input("\nApprove this output? (yes/no): ").lower()
    return {"approved": choice == "yes"}

def review_route(state):
    if state["approved"]:
        return "final"
    return "regenerate"

def final_output(state):
    print("\nFinal output:\n", state["draft"])
    return {"final": state["draft"]}

builder = StateGraph(ApprovalState)
builder.add_node("generate", generate_output)
builder.add_node("review", review_output)
builder.add_node("regenerate", generate_output)
builder.add_node("final", final_output)

builder.add_edge(START, "generate")
builder.add_edge("generate", "review")

builder.add_conditional_edges(
    "review",
    review_route,
    {"final": "final", "regenerate": "regenerate"}
)

builder.add_edge("regenerate", "review")
builder.add_edge("final", END)

approval_graph = builder.compile()

topic = input("Topic: ")

approval_graph.invoke({
    "topic": topic,
    "draft": "",
    "approved": False,
    "final": ""
})

# 18. Multi-agent Collaboration Graph

Two agents are represented as two nodes: Research Agent → Summary Agent.

In [ ]:
class AgentState(TypedDict):
    topic: str
    research: str
    summary: str

def research_agent(state):
    research_prompt = PromptTemplate(
        input_variables=["topic"],
        template="""Research the topic below using your general knowledge.
Give 5 useful points.

Topic: {topic}"""
    )

    research = (research_prompt | llm).invoke({
        "topic": state["topic"]
    }).content

    return {"research": research}

def summary_agent(state):
    summary_prompt = PromptTemplate(
        input_variables=["research"],
        template="""Summarize the research into a short final response.

Research:
{research}"""
    )

    summary = (summary_prompt | llm).invoke({
        "research": state["research"]
    }).content

    return {"summary": summary}

builder = StateGraph(AgentState)
builder.add_node("research_agent", research_agent)
builder.add_node("summary_agent", summary_agent)

builder.add_edge(START, "research_agent")
builder.add_edge("research_agent", "summary_agent")
builder.add_edge("summary_agent", END)

multi_agent_graph = builder.compile()

topic = input("Topic to research: ")
result = multi_agent_graph.invoke({
    "topic": topic,
    "research": "",
    "summary": ""
})

print("RESEARCH:\n", result["research"])
print("\nSUMMARY:\n", result["summary"])

# 19. Customer Support Workflow

The graph includes Greeting, Intent Detection, FAQ Retrieval, Escalation and Ticket Generation.

In [ ]:
class SupportState(TypedDict):
    query: str
    intent: str
    faq: str
    response: str

def greeting(state):
    print("Hello! Welcome to customer support.")
    return {}

def detect_support_intent(state):
    q = state["query"].lower()

    if any(x in q for x in ["refund", "payment", "charged"]):
        return {"intent": "billing"}
    elif any(x in q for x in ["broken", "error", "not working"]):
        return {"intent": "technical"}
    elif any(x in q for x in ["complaint", "manager", "escalate"]):
        return {"intent": "escalation"}
    return {"intent": "faq"}

def faq_retrieval(state):
    faq_data = {
        "faq": "Our support team is available during business hours.",
        "billing": "For billing issues, please check your payment details.",
        "technical": "Please restart the application and try again."
    }
    return {"faq": faq_data.get(state["intent"], faq_data["faq"])}

def escalation(state):
    return {
        "faq": "Your issue needs to be escalated to a support representative."
    }

def ticket_generation(state):
    response = f"Ticket created for {state['intent']} issue. Details: {state['faq']}"
    return {"response": response}

def support_route(state):
    if state["intent"] == "escalation":
        return "escalation"
    return "faq"

builder = StateGraph(SupportState)
builder.add_node("greeting", greeting)
builder.add_node("intent", detect_support_intent)
builder.add_node("faq", faq_retrieval)
builder.add_node("escalation", escalation)
builder.add_node("ticket", ticket_generation)

builder.add_edge(START, "greeting")
builder.add_edge("greeting", "intent")
builder.add_conditional_edges(
    "intent",
    support_route,
    {"faq": "faq", "escalation": "escalation"}
)
builder.add_edge("faq", "ticket")
builder.add_edge("escalation", "ticket")
builder.add_edge("ticket", END)

support_graph = builder.compile()

query = input("Customer issue: ")
result = support_graph.invoke({
    "query": query,
    "intent": "",
    "faq": "",
    "response": ""
})

print("\nIntent:", result["intent"])
print("Response:", result["response"])

# 20. End-to-end AI Assistant Graph (Capstone)

This combines the main ideas:
1. Accept user query
2. Classify query
3. Retrieve relevant information
4. Use a tool when required
5. Generate response
6. Collect feedback
7. Store feedback in graph state

In [ ]:
from typing import TypedDict, Any

class CapstoneState(TypedDict):
    query: str
    query_type: str
    context: str
    tool_result: str
    response: str
    feedback: str

def classify_query(state):
    q = state["query"].lower()

    if any(x in q for x in ["calculate", "add", "multiply", "divide", "+", "-", "*", "/"]):
        query_type = "math"
    elif any(x in q for x in ["python", "code", "programming"]):
        query_type = "coding"
    else:
        query_type = "general"

    return {"query_type": query_type}

def retrieve_documents(state):
    # Small sample knowledge base for the capstone demo.
    knowledge = {
        "coding": "Python is a popular programming language used for AI, data analysis and automation.",
        "general": "LangChain is used for building applications around language models.",
        "math": "Math questions can be handled using a calculator tool."
    }

    return {"context": knowledge.get(state["query_type"], "")}

def use_tool(state):
    if state["query_type"] == "math":
        expression = state["query"].lower()
        expression = expression.replace("calculate", "").strip()

        try:
            result = calculator.invoke(expression)
        except:
            result = "The calculator could not understand the expression."

        return {"tool_result": result}

    return {"tool_result": ""}

def generate_response(state):
    prompt = PromptTemplate(
        input_variables=["query", "context", "tool_result"],
        template="""Answer the user using the available information.

User query:
{query}

Context:
{context}

Tool result:
{tool_result}

Give a clear and short answer."""
    )

    response = (prompt | llm).invoke({
        "query": state["query"],
        "context": state["context"],
        "tool_result": state["tool_result"]
    }).content

    return {"response": response}

def collect_feedback(state):
    print("\nAssistant:", state["response"])
    feedback = input("Was this helpful? (yes/no): ")
    return {"feedback": feedback}

builder = StateGraph(CapstoneState)

builder.add_node("classify", classify_query)
builder.add_node("retrieve", retrieve_documents)
builder.add_node("tool", use_tool)
builder.add_node("generate", generate_response)
builder.add_node("feedback", collect_feedback)

builder.add_edge(START, "classify")
builder.add_edge("classify", "retrieve")
builder.add_edge("retrieve", "tool")
builder.add_edge("tool", "generate")
builder.add_edge("generate", "feedback")
builder.add_edge("feedback", END)

capstone_graph = builder.compile()

query = input("Enter your query: ")

result = capstone_graph.invoke({
    "query": query,
    "query_type": "",
    "context": "",
    "tool_result": "",
    "response": "",
    "feedback": ""
})

print("\nStored feedback:", result["feedback"])

## Assignment completed

All 20 practical questions from the provided LangChain & LangGraph assignment are included in this notebook.